# Demo de factibilidad de metadata para ruteo resiliente ante inundaciones urbanas

Este notebook demuestra que es posible obtener:
- Infraestructura vial desde OpenStreetMap (Overpass API)
- Metadata de elevación (Open-Meteo Elevation API)
- Metadata de lluvia (Open-Meteo Weather API)
- Amenazas: estaciones hidrométricas DGA (GeoJSON)

Todo mediante peticiones HTTP automatizables.

In [2]:
import requests
import json
from pprint import pprint
import pandas as pd

try:
    import geopandas as gpd
except ImportError:
    gpd = None
    print("Geopandas no está instalado. Usa: pip install geopandas")

Geopandas no está instalado. Usa: pip install geopandas


## 1. Configuración inicial

In [3]:
CITY_NAME = "Concepción"
BBOX = [-36.90, -73.10, -36.70, -72.90]
print("Ciudad:", CITY_NAME)
print("BBOX:", BBOX)

Ciudad: Concepción
BBOX: [-36.9, -73.1, -36.7, -72.9]


## 2. Infraestructura: red vial desde OSM con Overpass API

In [4]:
def fetch_osm_roads(bbox, highway_filter=None):
    min_lat, min_lon, max_lat, max_lon = bbox
    overpass_url = "https://overpass-api.de/api/interpreter"

    if highway_filter:
        highway_clause = f'["highway"~"{highway_filter}"]'
    else:
        highway_clause = '["highway"]'

    query = f"""
    [out:json][timeout:60];
    (
      way{highway_clause}({min_lat},{min_lon},{max_lat},{max_lon});
    );
    (._;>;);
    out body;
    """

    response = requests.post(overpass_url, data={"data": query})
    response.raise_for_status()
    return response.json()

osm_data = fetch_osm_roads(BBOX, highway_filter="primary|secondary|residential")
print("Elementos OSM descargados:", len(osm_data.get("elements", [])))
pprint(osm_data.get("elements", [])[:5])

Elementos OSM descargados: 41249
[{'id': 267252253, 'lat': -36.8266991, 'lon': -73.0403207, 'type': 'node'},
 {'id': 267252256,
  'lat': -36.8267388,
  'lon': -73.0403424,
  'tags': {'crossing': 'uncontrolled',
           'crossing:markings': 'zebra',
           'highway': 'crossing'},
  'type': 'node'},
 {'id': 267252355, 'lat': -36.8262151, 'lon': -73.0373863, 'type': 'node'},
 {'id': 267293828, 'lat': -36.7540448, 'lon': -73.0010545, 'type': 'node'},
 {'id': 267293907,
  'lat': -36.7527022,
  'lon': -73.0004999,
  'tags': {'highway': 'motorway_junction', 'name': 'Enlace Penco'},
  'type': 'node'}]


## 3. Metadata: elevación desde Open-Meteo Elevation API

In [5]:
def extract_sample_nodes(osm_json, max_nodes=5):
    nodes = [e for e in osm_json.get("elements", []) if e.get("type") == "node"]
    sample = nodes[:max_nodes]
    coords = [(n["lat"], n["lon"]) for n in sample]
    return coords

sample_coords = extract_sample_nodes(osm_data)
sample_coords[:5]

[(-36.8266991, -73.0403207),
 (-36.8267388, -73.0403424),
 (-36.8262151, -73.0373863),
 (-36.7540448, -73.0010545),
 (-36.7527022, -73.0004999)]

In [6]:
def fetch_elevation(coords):
    if not coords:
        raise ValueError("No hay coordenadas para consultar elevación.")

    lats = ",".join(str(c[0]) for c in coords)
    lons = ",".join(str(c[1]) for c in coords)

    url = "https://api.open-meteo.com/v1/elevation"
    params = {"latitude": lats, "longitude": lons}
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()

elev = fetch_elevation(sample_coords)
print("Respuesta elevación:")
pprint(elev)

Respuesta elevación:
{'elevation': [20.0, 20.0, 16.0, 56.0, 62.0]}


## 4. Metadata: precipitación desde Open-Meteo Weather API

In [7]:
lat_center = (BBOX[0] + BBOX[2]) / 2
lon_center = (BBOX[1] + BBOX[3]) / 2

def fetch_precip(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "precipitation",
        "timezone": "auto"
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()

rain = fetch_precip(lat_center, lon_center)
pprint(rain.get("hourly", {}).get("precipitation", [])[:10])

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


## 5. Amenazas: estaciones hidrométricas DGA (GeoJSON)

In [8]:
import io, zipfile

def fetch_dga():
    url = "https://lineasdebasepublicas.mma.gob.cl/datos_abiertos/dataset/bdc95936-1ea3-4625-985b-8885a10812d6/resource/d18e82ca-85c2-4ef6-ad4d-a91c9cbc7b57/download/estacion-de-red-hidrometrica-dga_geojson.zip"
    r = requests.get(url)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    geo = [n for n in z.namelist() if n.endswith(".geojson")][0]
    with z.open(geo) as f:
        data = json.loads(f.read().decode("utf-8"))
    return data

dga = fetch_dga()
print("Total estaciones:", len(dga.get("features", [])))
pprint(dga.get("features", [])[:2])

Total estaciones: 3425
[{'geometry': {'coordinates': [-68.58332002799995, -22.333628502999943],
               'type': 'Point'},
  'properties': {'altitud': 2550,
                 'cod_epsg': 4326,
                 'cod_estacion': '02105013',
                 'comuna': 'Calama',
                 'latitud': -22.333628502999943,
                 'longitud': -68.58332002799995,
                 'lugar': 'Cauce Natural',
                 'nombre': 'Laguna Chiu-Chiu (Ca)',
                 'objectid': 118,
                 'region': 'Antofagasta',
                 'sscuenca_dga': '02105',
                 'tipo_estacion': 'calidad de agua'},
  'type': 'Feature'},
 {'geometry': {'coordinates': [-70.76723238499994, -28.571119833999944],
               'type': 'Point'},
  'properties': {'altitud': 0,
                 'cod_epsg': 4326,
                 'cod_estacion': '03823014',
                 'comuna': 'Vallenar',
                 'latitud': -28.571119833999944,
                 'longitud':

## 6. Filtrar estaciones dentro del bounding box

In [9]:
def filter_bbox(features, bbox):
    min_lat, min_lon, max_lat, max_lon = bbox
    out = []
    for f in features:
        geom = f.get("geometry", {})
        if geom.get("type") == "Point":
            lon, lat = geom.get("coordinates", [None, None])
            if lat and lon:
                if min_lat <= lat <= max_lat and min_lon <= lon <= max_lon:
                    out.append(f)
    return out

filtered = filter_bbox(dga.get("features", []), BBOX)
print("Estaciones en bbox:", len(filtered))
pprint(filtered[:5])

Estaciones en bbox: 15
[{'geometry': {'coordinates': [-73.01657292999994, -36.817267690999984],
               'type': 'Point'},
  'properties': {'altitud': 0,
                 'cod_epsg': 4326,
                 'cod_estacion': '08220004',
                 'comuna': 'Florida',
                 'latitud': -36.817267690999984,
                 'longitud': -73.01657292999994,
                 'lugar': 'Cauce Natural',
                 'nombre': 'Rio Andalien Antes Estero Nonguen',
                 'objectid': 1491,
                 'region': 'Biobío',
                 'sscuenca_dga': '08220',
                 'tipo_estacion': 'calidad de agua'},
  'type': 'Feature'},
 {'geometry': {'coordinates': [-73.01659962999997, -36.81579898999996],
               'type': 'Point'},
  'properties': {'altitud': 17,
                 'cod_epsg': 4326,
                 'cod_estacion': '08220002',
                 'comuna': 'Florida',
                 'latitud': -36.81579898999996,
                 'longit

## Conclusión

Todas las fuentes funcionan correctamente y devuelven datos automatizables:
- Red vial desde OSM vía Overpass
- Elevación desde Open-Meteo
- Lluvia desde Open-Meteo
- Estaciones hidrométricas DGA como amenaza

Esto confirma la factibilidad técnica de la Fase 1.